# 07 — Temporal Transformer Autoencoder (Etapa 6)

Sequências de 9 trimestres (t−8 … t) de cada **célula de 320 m** (embedding TerraMind S2 + S1), com codificação de
ano e de trimestre (seno/cosseno) e máscara para trimestres sem dado. O modelo **nunca vê Z(t)** ao produzir Ẑ(t):

- **forecast:** o passo t é sempre escondido; prevê Z(t) a partir de t−8 … t−1;
- **masked:** no treino, além de t, ~25% dos passos de contexto são escondidos e reconstruídos.

As duas pontuam igual: esconde t, mede o erro. **A formulação principal é escolhida pela perda nos normais de
validação** (sem rótulos de anomalia). As duas entram na tabela como o E10.

Score da célula = erro quadrático de Ẑ(t) (embeddings padronizados). Patch = **máximo** das 49 células.
**Normalização local (PLANO 6.4):** z-score do erro pelos erros das células normais de validação da mesma cobertura,
para que pastagem (erro "normal" maior) não vire o mapa inteiro.

Treino só com amostras normais (regra por janela) de `train`, T ≤ 2022. Mesma avaliação do `06_baselines`.

**Tempo em CPU (medido em 29/10, d_model = 128):** treino ~1,5 min por 100 mil sequências por época; pontuação
~150 s por milhão de células. Sub-AOI: ~30 min no total. Tapajós completo: ~75 min por formulação + ~25 min de pontuação.

In [ ]:
import sys, os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    raise RuntimeError('Fluxo local desde 29/10 (ver "Limitações de infraestrutura" no PLANO). '
                       'No Colab, siga o setup do 03_cubo_sentinel e use /content/cache para os pixels.')
import json, time
import numpy as np
import pandas as pd
import config, utils, rotulos, dados, embeddings
config.make_dirs()
utils.set_seed(config.SEED)

import torch, avaliacao, baselines, modelo
FONTE, AOI = 'tapajos_sub', 'tapajos'        # depois: ('tapajos', 'tapajos'); norte_ro só pontua (carrega o checkpoint)
EVAL_SPLIT = 'train' if FONTE.endswith('_sub') else ('test' if AOI == 'tapajos' else 'test_gen')
N_TRAIN, N_VAL = 200_000, 50_000              # sequências de células sorteadas
D_MODEL, EPOCHS = 128, 40
print('avaliação em', EVAL_SPLIT, '(DIAGNÓSTICO, não reportar)' if EVAL_SPLIT == 'train' else '')
run_dir = utils.new_run(f'etapa6_transformer_ae_{FONTE}')

## 1. Sequências de treino e validação (só normais)

In [ ]:
s, ctx, lab = baselines.load(FONTE, AOI, k=modelo.K_SEQ)
tr = modelo.gather_training(s, ctx, 'treino_normal', N_TRAIN)
va = modelo.gather_training(s, ctx, 'val_normal', N_VAL, seed=config.SEED + 1)
print('treino', tr[0].shape, '| validação', va[0].shape, '| amostras a pontuar', int((s['t'] >= modelo.K_SEQ).sum()))

## 2. Treino das duas formulações

In [ ]:
trainers = {}
for mode in ['forecast', 'masked']:
    print(f'== {mode}')
    trainers[mode] = modelo.Trainer(*tr, *va, mode=mode, d_model=D_MODEL, epochs=EPOCHS)
    trainers[mode].save(config.CHECKPOINTS_DIR / f'transformer_ae_{mode}_{FONTE}.pt')
    trainers[mode].save(run_dir / f'transformer_ae_{mode}.pt')
best_mode = min(trainers, key=lambda m: trainers[m].best_val)
print({m: round(t.best_val, 4) for m, t in trainers.items()}, '-> formulação principal:', best_mode)

In [ ]:
import matplotlib.pyplot as plt
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e1e0d9'
CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']      # slots categóricos na ordem fixa
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': GRID, 'axes.labelcolor': MUTED, 'xtick.color': MUTED,
                     'ytick.color': MUTED, 'axes.spines.top': False, 'axes.spines.right': False})

fig, ax = plt.subplots(figsize=(8, 3.4))
for (mode, t), c in zip(trainers.items(), CAT):
    cv = pd.DataFrame(t.curve)
    ax.plot(cv['epoca'], cv['treino'], color=c, lw=1.5, ls='--', label=f'{mode} treino')
    ax.plot(cv['epoca'], cv['val'], color=c, lw=2, label=f'{mode} validação (erro em t)')
    pd.DataFrame(t.curve).to_csv(run_dir / f'curva_{mode}.csv', index=False)
ax.set_xlabel('época'); ax.set_ylabel('erro quadrático (padronizado)'); ax.grid(axis='y', color=GRID, lw=0.5)
ax.legend(frameon=False, fontsize=8); ax.set_title('Curvas de perda', color=INK)
fig.tight_layout(); fig.savefig(run_dir / 'curvas.png', dpi=130); plt.show()

## 3. Pontuação de todas as células

In [ ]:
scores = []
for mode, t in trainers.items():
    t0 = time.time()
    err = modelo.score_all(t, s, ctx)
    scores.append(modelo.patch_scores(err, s, f'Transformer AE | {mode}'))
    print(f'{mode}: {(time.time() - t0) / 60:.1f} min')
scores = pd.concat(scores, axis=1)
rows = s.loc[scores.index]
pd.concat([rows[['patch_id', 'step']], scores], axis=1).to_csv(run_dir / 'scores_trimestrais.csv.gz', index=False)

## 4. Avaliação e comparação com os baselines

Mesmo protocolo e mesmas amostras do `06_baselines` (amostras com 8 trimestres de histórico, t ≥ 2021Q1). Os baselines
são lidos do último run do `06` para a mesma fonte e restritos às mesmas amostras.

In [ ]:
import glob
def eval_all(score_df, labels, key):
    out = []
    for col in [c for c in score_df.columns if '|' in c]:
        sc = score_df[['patch_id', key]].assign(score=score_df[col].values).dropna(subset=['score'])
        r = avaliacao.evaluate(sc, labels, lab['grade'], key=key, split=EVAL_SPLIT, n_boot=200)
        out.append(r.assign(metodo=col))
    return pd.concat(out, ignore_index=True)

b_run = sorted(glob.glob(str(config.RESULTS_DIR / f'etapa5_baselines_{FONTE}' / '*' / 'scores_trimestrais.csv.gz')))[-1]
base = pd.read_csv(b_run)
q = rows[['patch_id', 'step']].join(scores).merge(base, on=['patch_id', 'step'], how='left')
res_q = eval_all(q, lab['trim'], 'step').assign(resolucao='trimestral')
q['year'] = q['step'].str[:4].astype(int)
a = q.drop(columns='step').groupby(['patch_id', 'year']).max().reset_index()
res_a = eval_all(a, lab['anual'], 'year').assign(resolucao='anual')
res = pd.concat([res_q, res_a], ignore_index=True)
res['agregacao'] = np.where(res['metodo'].str.endswith('[média]'), 'média', 'máximo')
res['metodo'] = res['metodo'].str.replace(' [média]', '', regex=False)
res.to_csv(run_dir / 'metricas.csv', index=False)
print('baselines de', b_run)

In [ ]:
main = res[(res['agregacao'] == 'máximo') & (res['tarefa'] == 'todos')]
cols = ['AUROC', 'AUROC_lo', 'AUROC_hi', 'AUPRC', 'TPR@1%FPR', 'n_pos']
for resol in ['trimestral', 'anual']:
    t = main[main['resolucao'] == resol].set_index('metodo')[cols].sort_values('AUROC', ascending=False)
    print(f'== {resol}, todas as anomalias ({EVAL_SPLIT}, 2023–2025)'); display(t.head(12).round(3))
print('== AUROC por tipo, trimestral')
display(res[(res['agregacao'] == 'máximo') & (res['resolucao'] == 'trimestral') & res['metodo'].str.startswith('Transformer')]
        .pivot(index='metodo', columns='tarefa', values='AUROC').round(3))

## 5. E2: temporal vs. estático, e o efeito da normalização local

A pergunta que justifica o modelo: o Transformer AE supera o **kNN estático** (só Z(t))? E a normalização por cobertura
reduz a vantagem "é fronteira?" que o estático tem na avaliação anual?

In [ ]:
import matplotlib.pyplot as plt
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e1e0d9'
CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']      # slots categóricos na ordem fixa
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': GRID, 'axes.labelcolor': MUTED, 'xtick.color': MUTED,
                     'ytick.color': MUTED, 'axes.spines.top': False, 'axes.spines.right': False})

keys = [f'Transformer AE | {best_mode}', f'Transformer AE | {best_mode} + norm. local',
        'TerraMind S2S1 | kNN estático (sem tempo)', 'TerraMind S2S1 | kNN no delta', 'Espectral | queda de NDVI']
d = main[main['metodo'].isin(keys)]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharex=True)
for ax, resol in zip(axes, ['trimestral', 'anual']):
    x = d[d['resolucao'] == resol].set_index('metodo').reindex(keys).reset_index()
    y = np.arange(len(x))[::-1]
    for i, (yy, r) in enumerate(zip(y, x.itertuples())):
        c = CAT[0] if r.metodo.startswith('Transformer') else CAT[1]
        ax.errorbar(r.AUROC, yy, xerr=[[r.AUROC - r.AUROC_lo], [r.AUROC_hi - r.AUROC]], fmt='o', ms=7, color=c,
                    ecolor=c, elinewidth=1.5)
    ax.set_yticks(y, x['metodo'], fontsize=8); ax.axvline(0.5, color=GRID, ls='--', lw=1)
    ax.grid(axis='x', color=GRID, lw=0.5); ax.set_title(f'{resol}: todas as anomalias', color=INK)
    ax.set_xlabel('AUROC (IC 95%)')
fig.tight_layout(); fig.savefig(run_dir / 'e2_temporal_vs_estatico.png', dpi=130); plt.show()

In [ ]:
(run_dir / 'resumo.json').write_text(json.dumps({
    'fonte': FONTE, 'eval_split': EVAL_SPLIT, 'formulacao_principal': best_mode,
    'val_loss': {m: t.best_val for m, t in trainers.items()}, 'config': trainers[best_mode].cfg,
    'epocas': {m: len(t.curve) for m, t in trainers.items()},
    'auroc_todos': main.set_index(['resolucao', 'metodo'])['AUROC'].round(4).unstack(0).to_dict()},
    indent=2, default=str), encoding='utf-8')
print('Salvo em', run_dir)